# 02 — 离散隐变量：直接 MLE
沿用第 01 章的二维数据与 MLE 目标，只改变概率模型：不再直接假设观测来自单个高斯，而是假设先抽取类别 $x\in\{0,1,2\}$，再从该类别的高斯分布生成观测。

主动定义 $p(x=k)=\pi_k$ 与 $p(y|x=k)=\mathcal N(y;\mu_k,\sigma^2I)$。固定 $\sigma=0.45$；待学习参数是类别概率 $\pi$ 与三个二维均值 $\mu_k$。类别标签不可观测。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, multivariate_normal
from utils import generate_observations, compare, plot_history

# 共享数据，只观测 y；不读取真实生成类别。
rng = np.random.default_rng(42)
y = generate_observations(rng)

## 1. 给定初始参数，计算五个概率量
先不训练。定义先验和条件分布，乘法得到联合密度，对类别求和得到边缘密度，最后可选地计算后验。后验用于隐变量推断。

In [ ]:
# 固定观测噪声；mu0、pi0 是人为选择的初始参数，不是真值。
sigma = .45
mu0 = np.array([[-1., 0.], [1., 0.], [0., 2.]])
pi0 = np.ones(3) / 3

def probabilities(y, mu, pi):
    N = len(y)
    K = len(pi)
    # prior[k] = p(x=k)，由当前参数直接确定。
    prior = pi.copy()
    conditional = np.zeros((N, K))
    joint = np.zeros((N, K))
    marginal = np.zeros(N)
    posterior = np.zeros((N, K))

    # 第 k 个条件分布 p(y|x=k)，由当前均值和固定协方差定义。
    models = []
    for k in range(K):
        models.append(multivariate_normal(mu[k], sigma**2 * np.eye(2)))

    # 对每个观测样本，逐个枚举隐变量的取值。
    for i in range(N):
        for k in range(K):
            conditional[i, k] = models[k].pdf(y[i])
            # p(y_i,x=k) = p(y_i|x=k) p(x=k)
            joint[i, k] = conditional[i, k] * prior[k]

        # 对隐变量求和：p(y_i)
        for k in range(K):
            marginal[i] += joint[i, k]

        # 贝叶斯公式：p(x=k | y_i)
        for k in range(K):
            posterior[i, k] = joint[i, k] / marginal[i]

    return prior, conditional, joint, marginal, posterior

parts = probabilities(y, mu0, pi0)
prior, conditional, joint, marginal, posterior = parts
print('prior:', prior)
print('conditional:', conditional[0])
print('joint:', joint[0])
print('marginal:', marginal[0])
print('posterior:', posterior[0])

for i in range(len(y)):
    assert np.isclose(sum(parts[-1][i]), 1)

## 2. 沿用第 01 章的最大似然目标
仍然最大化 $\sum_i\log p_\theta(y_i)$，但这里 $p_\theta(y_i)=\sum_k p_\theta(y_i,x=k)$。不同分量出现在 log 内部的求和中，一般不能像第 01 章那样直接求出解析参数解。改用 PyTorch 梯度优化。

logits 是三个无约束的实数，经 softmax 得到非负且和为 1 的类别概率。logsumexp 计算 $\log\sum_k\exp(a_k)$，与先求和再取 log 等价，但数值更稳定。

In [ ]:
import torch
torch.set_num_threads(1)
torch.manual_seed(42)
Y = torch.tensor(y, dtype=torch.float64)
mu = torch.nn.Parameter(torch.tensor(mu0.copy()))
logits = torch.nn.Parameter(torch.zeros(3, dtype=torch.float64))
R = sigma**2 * torch.eye(2, dtype=Y.dtype)
opt = torch.optim.Adam([mu, logits], lr=.05)

def log_marginal(Y, mu, logits):
    # 定义先验：log_pi[k] = log p(x=k)
    log_pi = logits.log_softmax(dim=0)
    # 每一列保存一个类别对应的所有样本联合对数密度。
    columns = []
    for k in range(len(pi0)):
        conditional = torch.distributions.MultivariateNormal(mu[k], R)
        log_conditional = conditional.log_prob(Y)
        log_joint = log_conditional + log_pi[k]
        columns.append(log_joint)
    # 形状为 [样本数, 类别数]；每行对应一个观测。
    joint = torch.stack(columns, dim=1)
    # 沿类别方向求和（在 log 空间实现）；返回每个样本的 log p(y_i)。
    return torch.logsumexp(joint, dim=1)

mle_history = []
for epoch in range(150):
    # 最大化平均观测对数似然。
    ll = log_marginal(Y, mu, logits).mean()
    loss = -ll
    opt.zero_grad()
    loss.backward()
    opt.step()
    mle_history.append(ll.item())

mu_mle = mu.detach().numpy()
pi_mle = logits.softmax(dim=0).detach().numpy()
print('means:', mu_mle)
print('prior:', pi_mle)
plot_history(mle_history)

## 3. 从训练完成的模型生成数据
先从学习到的类别先验抽取 $x$，再从对应条件分布抽取 $y$。这里不需要后验。

In [ ]:
samples = []
for i in range(len(y)):
    k = rng.choice(len(pi_mle), p=pi_mle)
    conditional = multivariate_normal(mu_mle[k], sigma**2*np.eye(2))
    samples.append(conditional.rvs(random_state=rng))
compare(y, np.array(samples))

## 小结
引入离散隐变量并未改变 MLE 目标。只要能计算边缘密度，就可以直接训练。下一章只将隐变量的取值从三个类别改成连续标量：对隐变量的求和将变成积分。